# Training

### Libraries and constants

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path
from joblib import parallel_backend
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

In [2]:
# --- Settings ---
pd.set_option('display.float_format', lambda x: f'{x:.6f}') # turn off exponent format (1e-12) for data frames, use float format instead

In [3]:
# --- CONSTANTS ---
RANDOM_SEED = 42
TARGET_NAME = 'state'

## Data import

In [5]:
df_train = pd.read_csv(Path('data') / '3 df_train_post_EDA.csv')
df_test = pd.read_csv(Path('data') / '3 df_test_post_EDA.csv')

print(f"df_train.shape = {df_train.shape}")
print(f"df_test.shape = {df_test.shape}")

df_train.shape = (5169, 42)
df_test.shape = (1293, 42)


In [6]:
X_train, y_train = df_train.drop(columns=[TARGET_NAME]), df_train[TARGET_NAME]
X_test, y_test = df_test.drop(columns=[TARGET_NAME]), df_test[TARGET_NAME]

## Training

### SVM

The Tuning Philosophy: SVMs are mathematically rigid. Their performance hinges heavily on a small set of interacting continuous parameters—primarily the regularization parameter ($C$) and the kernel coefficient ($\gamma$). Because the parameter space is relatively small but highly sensitive to specific combinations, we use GridSearchCV to exhaustively test every specified combination.

As we stated in EDA, we need a regularization parameter to account for slight geometric distortion. In SVM, this regularization happens to be $C$.

In [ ]:
# 1. Define your pipeline and grid
svm_pipeline = Pipeline([
    ('svm', SVC(
                class_weight='balanced', 
                random_state=RANDOM_SEED,
                probability=False, # set to True if you need probability outputs later (SHAP, calibration, or any predict_proba call). It costs training time but must be set before fitting — cannot be added after
            )
    ),
])

svm_param_grid = [
    {
        'svm__kernel': ['linear'],
        'svm__C': [0.001, 0.01, 0.1, 1, 10],   # log scale; small C = strong L2
    },
    {
        'svm__kernel': ['rbf'],
        'svm__C': [0.1, 1, 10, 100],
        'svm__gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    },
    {
        'svm__kernel': ['poly'],
        'svm__C': [0.1, 1, 10],
        'svm__degree': [2, 3],
    },
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# 2. Setup grid search with verbose=3
svm_grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_param_grid,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=3,
)

# 3. Fit the model using threading backend so prints show up live
print("Starting grid search...")
with parallel_backend('threading'):
    svm_grid_search.fit(X_train, y_train)

# 4. Print the best training results
print(f"\nBest estimator: {svm_grid_search.best_estimator_}\n")
print(f"Best SVM Parameters: {svm_grid_search.best_params_}")
print(f"Best SVM CV Score: {svm_grid_search.best_score_:.4f}\n")

# 5. Predict on the test dataset
print("Evaluating on test data...")
y_pred = svm_grid_search.predict(X_test)

print("\n=== TEST SET CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

Starting grid search...
Fitting 5 folds for each of 31 candidates, totalling 155 fits
[CV 3/5] END ...svm__C=0.01, svm__kernel=linear;, score=0.787 total time=   1.3s
[CV 2/5] END ...svm__C=0.01, svm__kernel=linear;, score=0.783 total time=   1.4s
[CV 1/5] END ...svm__C=0.01, svm__kernel=linear;, score=0.780 total time=   1.4s
[CV 5/5] END ..svm__C=0.001, svm__kernel=linear;, score=0.734 total time=   2.2s
[CV 1/5] END ..svm__C=0.001, svm__kernel=linear;, score=0.727 total time=   2.3s
[CV 1/5] END ....svm__C=0.1, svm__kernel=linear;, score=0.829 total time=   0.8s
[CV 4/5] END ..svm__C=0.001, svm__kernel=linear;, score=0.722 total time=   2.3s
[CV 3/5] END ..svm__C=0.001, svm__kernel=linear;, score=0.721 total time=   2.4s
[CV 2/5] END ..svm__C=0.001, svm__kernel=linear;, score=0.724 total time=   2.5s
[CV 4/5] END ...svm__C=0.01, svm__kernel=linear;, score=0.796 total time=   1.1s
[CV 5/5] END ...svm__C=0.01, svm__kernel=linear;, score=0.788 total time=   1.1s
[CV 2/5] END ....svm__C

As presumed during visualization, linear models show the poorest performance among the three. The predictive power of the best model is impressively high:
```txt
Best SVM Parameters: {'svm__C': 100, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Best SVM CV Score: 0.9746
=== TEST SET CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           1       1.00      1.00      1.00       682
           2       0.99      0.99      0.99       370
           3       1.00      0.98      0.99       208
           4       1.00      1.00      1.00        33

    accuracy                           0.99      1293
   macro avg       0.99      0.99      0.99      1293
weighted avg       0.99      0.99      0.99      1293
```

$C=100$ for RBF is normal for $\kappa=20.56$. RBF maps the data into a (effectively infinite-dimensional) kernel space where the original geometric distortion is irrelevant — the kernel recomputes all distances from scratch. Large C for RBF is completely normal and not a sign of overfitting in the same way it would be for a linear SVM.

### Trees

The Tuning Philosophy: Random Forests have a massive hyperparameter space (n_estimators, max_depth, min_samples_split, max_features, etc.). Running an exhaustive grid search on a forest is computationally crippling and mathematically unnecessary. Research shows that testing a random sample of parameter combinations (RandomizedSearchCV) yields near-optimal results in a fraction of the time, because only a few hyperparameters truly dictate the model's success.

In [11]:
rf_pipeline = Pipeline([
    ('rf', RandomForestClassifier(
                class_weight="balanced_subsample", # Each tree gets its own unique bootstrap sample of data. The algorithm recalculates the balanced weights based only on the rows selected for that specific tree.
                random_state=RANDOM_SEED
            )
    ),
])

rf_param_dist = {
    'rf__n_estimators': np.arange(100, 1001, 100),
    'rf__max_depth': [None, 10, 20, 30, 40, 50],
    'rf__min_samples_split': np.arange(2, 21, 2),
    'rf__min_samples_leaf': np.arange(1, 11, 2),
    'rf__max_features': ['sqrt', 'log2', 0.3, 0.5],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

rf_random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_dist,
    n_iter=80,          # 80 random combinations (~0.7%) to run (out of 10 × 6 × 10 × 5 × 4 = 12,000 options)
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=3,
)

print("Starting randomized grid search...")
with parallel_backend('threading'):
    rf_random_search.fit(X_train, y_train)

print(f"Best estimator: {rf_random_search.best_estimator_}\n")
print(f"Best RF Parameters: {rf_random_search.best_params_}")
print(f"Best RF CV Score: {rf_random_search.best_score_:.4f}\n")

print("Evaluating on test data...")
y_pred = rf_random_search.predict(X_test)

print("\n=== TEST SET CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

Starting randomized grid search...
Fitting 5 folds for each of 80 candidates, totalling 400 fits
[CV 1/5] END rf__max_depth=None, rf__max_features=log2, rf__min_samples_leaf=7, rf__min_samples_split=14, rf__n_estimators=100;, score=0.914 total time=   2.9s
[CV 3/5] END rf__max_depth=None, rf__max_features=log2, rf__min_samples_leaf=7, rf__min_samples_split=14, rf__n_estimators=100;, score=0.919 total time=   3.0s
[CV 2/5] END rf__max_depth=None, rf__max_features=log2, rf__min_samples_leaf=7, rf__min_samples_split=14, rf__n_estimators=100;, score=0.939 total time=   3.0s
[CV 5/5] END rf__max_depth=30, rf__max_features=0.3, rf__min_samples_leaf=5, rf__min_samples_split=16, rf__n_estimators=100;, score=0.878 total time=   5.7s
[CV 2/5] END rf__max_depth=30, rf__max_features=0.3, rf__min_samples_leaf=5, rf__min_samples_split=16, rf__n_estimators=100;, score=0.940 total time=   5.8s
[CV 3/5] END rf__max_depth=30, rf__max_features=0.3, rf__min_samples_leaf=5, rf__min_samples_split=16, rf__n_

Trees didn't surpass the radial SVM. Approximating a smooth hyperspherical boundary with rectangles requires many splits, and each split introduces a small error at the boundary edges. With enough trees this converges, but it never achieves the same geometric precision as a kernel that directly computes spherical distances.
```txt
Best RF Parameters: {'rf__n_estimators': 600, 'rf__min_samples_split': 8, 'rf__min_samples_leaf': 5, 'rf__max_features': 'sqrt', 'rf__max_depth': 30}
Best RF CV Score: 0.9272

Evaluating on test data...

=== TEST SET CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           1       0.99      0.98      0.98       682
           2       0.98      0.97      0.98       370
           3       0.94      0.96      0.95       208
           4       0.94      0.97      0.96        33

    accuracy                           0.98      1293
   macro avg       0.96      0.97      0.97      1293
weighted avg       0.98      0.98      0.98      1293
```

## Conclusion

The scores have surpassed the ones in the paper, but the paper classified 7 activities including ambiguous overlapping ones like "standing up, walking and going up/down stairs" as a single class. We had 4 physiologically distinct activities — running and idle in particular are about as separable as two classes can be in accelerometer space.

The best model and scores:
```txt
Best model: {'svm__C': 100, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Test score: 0.99
```